# Этап 4. Собранный агент

В этом ноутбуке собраны проверяемые части финального слоя: MCP-клиент, хуки, ограничение контекста, журнал расходов и бесплатный просмотр сохранённых результатов. Новые обращения к модели из ячеек не запускаются автоматически.

## Цель и границы

Агент отвечает на вопросы о свежих событиях по локальному корпусу и живому поиску. MCP-серверы работают в Docker, а клиент подключается к ним по Streamable HTTP. Внешний текст считается данными, а решения о допустимых инструментах и размере результата принимает код.

In [ ]:
from pathlib import Path
import os

ROOT = Path.cwd()
print(ROOT)

## Набор задач

В наборе есть вопросы по базе, вопросы для живого поиска и вопросы без ответа. Разбиение нужно, чтобы не спутать качество RAG с качеством внешнего поиска.

In [ ]:
from collections import Counter
from events_agent import golden

tasks = golden.tasks()
print('всего задач:', len(tasks))
print(Counter(task['group'] for task in tasks))

## Хуки на границе инструментов

`before_tool` проверяет аргументы до сети, а `after_tool` ограничивает результат после сети. Это не просьба к модели соблюдать правило, а исполняемая проверка клиента.

In [ ]:
from events_agent.hooks import after_tool, before_tool

print(before_tool('web_search', {'query': '  2026 events  '}))
print(after_tool('web_search', 'result\x00 from a tool', 12))
try:
    before_tool('web_search', {'query': ' '})
except ValueError as error:
    print('проверка сработала:', error)

## Сжатие контекста

После нескольких обращений результаты инструментов могут занимать большую часть запроса. Код оставляет начало и последние полные раунды, а старые результаты сворачивает.

In [ ]:
from events_agent.context import compact, message_chars

messages = [{'role': 'system', 'content': 'stable'}, {'role': 'user', 'content': 'question'}]
for i in range(4):
    messages += [
        {'role': 'assistant', 'content': None, 'tool_calls': [{'id': str(i)}]},
        {'role': 'tool', 'tool_call_id': str(i), 'content': 'x' * 2000},
    ]
shorter, info = compact(messages, max_chars=2000)
print(info)
print(message_chars(messages), '->', message_chars(shorter))

## MCP-соединение

Следующая ячейка делает только `list_tools`: она не вызывает модель и не вызывает поисковые инструменты. Перед запуском должны быть здоровы `milvus`, `mcp` и `mcp-eval`.

In [ ]:
from events_agent import tools

mcp = tools.connect(os.getenv('MCP_URL', 'http://127.0.0.1:8090/mcp'))
try:
    print([tool.name for tool in mcp.tools])
finally:
    mcp.close()

## Бесплатная проверка результатов

Команда без `--yes` только читает существующие трейсы и печатает таблицу. Если записи нет, она сообщает, сколько задач ждёт отдельного согласования на новый вызов.

In [ ]:
import subprocess

result = subprocess.run(['uv', 'run', 'events-agent', 'eval'], text=True, capture_output=True)
print(result.stdout)
if result.stderr:
    print(result.stderr)

## Что проверено

- входные аргументы и результаты MCP проходят через кодовые хуки;
- история ограничивается до следующего вызова модели;
- для Claude подготовлены cache-control отметки, а журнал хранит долю входа из кэша;
- расходы ограничиваются на уровне задачи и календарного дня;
- MCP-инструменты выдаются по роли и сверяются с закреплёнными отпечатками.

Полное измерение запускается отдельной командой с `--yes` только после согласования расходов.